**02 | Signal Preprocessing: Windowing and Quality Controls**

**Human Movement Intelligence | Presented by: Reza Saadatyar (2026)**

**Objective:** Convert HARTH's continuous accelerometer recordings into fixed-length supervised activity windows using reusable Python code. Show the real output and explicitly reject windows crossing activity changes or timestamp gaps.

**Scope:** One participant, `S023`, for demonstration. No train/test split, fitted scaling, classification or performance claim is made.

**01. Import reusable functions and locate the source**

Reuse the data loader and visualiser from Notebook 01. The only new common function is `create_signal_windows` in `core/signal_preprocessing.py`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

root = Path.cwd().resolve()
while not (root / "code/python/src/core/data_loading.py").is_file():
    if root.parent == root:
        raise RuntimeError("Open the notebook inside the repository.")
    root = root.parent

sys.path.insert(0, str(root / "code/python/src"))
sys.path.insert(0, str(root / "studies/wearable_activity_recognition/src"))

from core.data_inspection import inspect_data, plot_signal
from core.signal_preprocessing import create_signal_windows
from harth_data import ACTIVITY_NAMES, ensure_harth_sample, load_harth_subject

print("Repository:", root)
print("Windowing function:", create_signal_windows.__name__)

**02. Load the same real HARTH recording**

The source file will be reused from the local ignored cache if Notebook 01 has already downloaded it.

In [ ]:
data_dir = root / "data" / "raw" / "harth_v2"
file_path = ensure_harth_sample(data_dir, subject="S023")
data, channels = load_harth_subject(file_path)
inspect_data(data, name="HARTH S023: continuous input")
print("Participant:", file_path.stem)
print("Recorded activity classes:", sorted(data["label"].unique().tolist()))

**03. Define a transparent windowing protocol**

At 50 Hz, 100 samples = 2 seconds; a step of 50 samples gives 50% overlap between candidate windows. Only windows where all 100 labels agree are kept. Any timestamps with a gap larger than 1.5 expected sample intervals are excluded within a candidate window.

In [ ]:
SAMPLING_RATE = 50
WINDOW_SECONDS = 2.0
OVERLAP = 0.5
WINDOW_SIZE = int(WINDOW_SECONDS * SAMPLING_RATE)
STEP_SIZE = int(WINDOW_SIZE * (1 - OVERLAP))

print(f"Sampling rate: {SAMPLING_RATE} Hz")
print(f"Window length: {WINDOW_SIZE} samples ({WINDOW_SECONDS:.1f} seconds)")
print(f"Step: {STEP_SIZE} samples | Overlap: {OVERLAP:.0%}")

**04. Run the reusable windowing function on actual signal values**

The output `X_windows` has the convention `(windows, channels, time)` for future PyTorch models. The `window_info` table preserves source start/end indices and the participant ID. `y_windows` contains one class per retained window.

In [ ]:
signals = data[channels].to_numpy()
labels = data["label"].to_numpy()

X_windows, y_windows, window_info = create_signal_windows(
    signals=signals,
    labels=labels,
    window_size=WINDOW_SIZE,
    step_size=STEP_SIZE,
    timestamps=data["timestamp"],
    sampling_rate=SAMPLING_RATE,
    min_label_purity=1.0,
    subject_id=file_path.stem,
)

print("Window tensor X:", X_windows.shape)
print("Window target y:", y_windows.shape)
print("First retained windows:")
display(window_info.head(8))

**05. Inspect activity distribution after windowing**

Compare the original class distribution with the retained windows. The counts need not preserve the original percentages, because strict label purity rejects windows around activity transitions.

In [ ]:
original_counts = data["label"].value_counts().sort_index()
window_counts = pd.Series(y_windows, name="label").value_counts().sort_index()
distribution = pd.DataFrame({
    "activity": [ACTIVITY_NAMES[int(code)] for code in original_counts.index],
    "source_samples": original_counts.to_numpy(),
    "retained_windows": [
        int(window_counts.get(code, 0)) for code in original_counts.index
    ],
})
display(distribution)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(distribution["activity"], distribution["retained_windows"])
ax.set(
    title="HARTH S023: retained single-activity windows",
    xlabel="Activity", ylabel="Number of 2-second windows",
)
ax.tick_params(axis="x", labelrotation=45)
fig.tight_layout()
plt.show()

**06. Visualise an actual preprocessed window**

This is a direct segment of a real recording, not a generated or filtered trace. The x-axis is time within a 2-second window. No normalisation has been applied.

In [ ]:
if len(X_windows) == 0:
    raise RuntimeError("No eligible windows; check source labels and timestamps.")

first_window = 0
activity_code = int(y_windows[first_window])
fig = plot_signal(
    X_windows[first_window, :3, :],
    title=f"HARTH S023: {ACTIVITY_NAMES[activity_code]} activity window",
    channel_names=channels[:3],
)
fig.axes[0].set_xlabel("Sample index within 2-second window (50 Hz)")
fig.axes[0].set_ylabel("Acceleration (g)")
plt.show()

print("Source indices:",
      int(window_info.iloc[first_window]["start_index"]), "to",
      int(window_info.iloc[first_window]["end_index_exclusive"]) - 1)
print("Label purity:", window_info.iloc[first_window]["label_purity"])

**07. Quality gate and interpretation**

The correct number and tensor shape are necessary but not sufficient for supervised learning. Every window in this notebook comes from **one participant**, so it is invalid to claim unseen-subject performance or to randomly split overlapping windows into Train and Test.

In [ ]:
assert X_windows.ndim == 3
assert X_windows.shape[1:] == (len(channels), WINDOW_SIZE)
assert len(X_windows) == len(y_windows) == len(window_info)
assert (window_info["label_purity"] == 1.0).all()
assert window_info["subject_id"].nunique() == 1

for row in window_info.itertuples(index=False):
    source_labels = labels[row.start_index:row.end_index_exclusive]
    assert len(source_labels) == WINDOW_SIZE
    assert (source_labels == row.label).all()

print("Window shape and source-label agreement: PASSED")
print("Subject-independent validation: NOT POSSIBLE WITH ONE PARTICIPANT")
print("Feature scaling/training completed: NO")
print("Next: evaluate preprocessing/features and a full multi-subject split.")